In [ ]:
# Ocean Listener — Dilation CNN for Underwater Bioacoustics
# Classes: whale / dolphin / ice
# Dataset folder structure:
# ocean_dataset/
#    whale/
#    dolphin/
#    ice/

import os
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from skimage.transform import resize

# -----------------------------
# PARAMETERS
# -----------------------------
DATASET_PATH = "ocean_dataset"
SAMPLE_RATE = 22050
SEGMENT_DURATION = 3
SAMPLES_PER_SEGMENT = SAMPLE_RATE * SEGMENT_DURATION
IMG_SIZE = (128,128)

# -----------------------------
# LABEL MAPPING
# -----------------------------
classes = sorted(os.listdir(DATASET_PATH))
class_to_index = {c:i for i,c in enumerate(classes)}

print("Classes:", classes)

# -----------------------------
# FEATURE EXTRACTION
# -----------------------------
def extract_segments(file_path):

    signal, sr = librosa.load(file_path, sr=SAMPLE_RATE)

    segments = []

    for start in range(0, len(signal), SAMPLES_PER_SEGMENT):

        end = start + SAMPLES_PER_SEGMENT

        if len(signal[start:end]) == SAMPLES_PER_SEGMENT:

            mel = librosa.feature.melspectrogram(
                y=signal[start:end],
                sr=sr,
                n_mels=128,
                n_fft=1024,
                hop_length=512
            )

            mel_db = librosa.power_to_db(mel, ref=np.max)

            mel_db = (mel_db - mel_db.min())/(mel_db.max()-mel_db.min()+1e-6)

            mel_img = resize(mel_db, IMG_SIZE)

            segments.append(mel_img)

    return segments

# -----------------------------
# LOAD DATASET
# -----------------------------
X=[]
y=[]

for cls in classes:

    folder=os.path.join(DATASET_PATH,cls)

    for file in os.listdir(folder):

        path=os.path.join(folder,file)

        try:

            segs=extract_segments(path)

            for s in segs:

                X.append(s)
                y.append(class_to_index[cls])

        except:
            print("Skipping:",path)
            continue

# -----------------------------
# CONVERT ARRAYS
# -----------------------------
X=np.array(X)
y=np.array(y)

X=X.reshape(-1,128,128,1)

print("Dataset shape:",X.shape)

# -----------------------------
# TRAIN TEST SPLIT
# -----------------------------
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42,stratify=y
)

# -----------------------------
# DILATION CNN MODEL
# -----------------------------
def dilation_block(x,filters,dilation):

    x=layers.Conv2D(
        filters,
        (3,3),
        padding="same",
        dilation_rate=dilation,
        activation="relu")(x)

    x=layers.BatchNormalization()(x)

    return x


def build_model():

    inp=layers.Input(shape=(128,128,1))

    x=dilation_block(inp,32,1)
    x=dilation_block(x,32,2)
    x=layers.MaxPool2D()(x)

    x=dilation_block(x,64,1)
    x=dilation_block(x,64,4)
    x=layers.MaxPool2D()(x)

    x=dilation_block(x,96,1)

    x=layers.GlobalAveragePooling2D()(x)

    x=layers.Dense(192,activation="relu")(x)
    x=layers.Dropout(0.4)(x)

    out=layers.Dense(3,activation="softmax")(x)

    model=models.Model(inp,out)

    return model

# -----------------------------
# BUILD MODEL
# -----------------------------
model=build_model()

model.summary()   # must be < 350k parameters

# -----------------------------
# COMPILE MODEL
# -----------------------------
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

# -----------------------------
# TENSORBOARD
# -----------------------------
logdir="logs/ocean_listener"

tb=tf.keras.callbacks.TensorBoard(logdir)

# -----------------------------
# TRAIN MODEL
# -----------------------------
history=model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.2,
    callbacks=[
        tb,
        tf.keras.callbacks.EarlyStopping(
            patience=8,
            restore_best_weights=True
        )
    ]
)

# -----------------------------
# ACCURACY PLOT
# -----------------------------
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend(["Train","Validation"])
plt.show()

# -----------------------------
# CONFUSION MATRIX
# -----------------------------
pred=np.argmax(model.predict(X_test),axis=1)

cm=confusion_matrix(y_test,pred)

disp=ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=classes
)

disp.plot(cmap="Blues")
plt.show()